# 02 · The Building Blocks: An Orchestration Engine in Plain Python

> **Kernel:** `Phase 5 · Concepts` · **Plain Python, runs offline.** No LLM is
> needed: the nodes are ordinary functions, so you can see the machinery itself.

Notebook 01 showed *why* a plain agent loop breaks. This notebook builds the fix:
a small but real orchestration engine, about 120 lines, that supports

- **state** with **reducers** (how updates are merged)
- **nodes** and **edges**, including **routers** (conditional edges)
- **parallel branches** (fan-out / fan-in)
- **cycles** with guards
- **checkpoints** in SQLite, **resume after a crash**, and separate runs (**threads**)
- **pause for a human** and resume later, in a different process
- **time travel**: inspect any past step and fork a new run from it

It's built in the same shape LangGraph uses internally (a "superstep" loop).
Every framework in this phase is some version of this machine. Once you've
built it, a framework's docs read as "that's their name for a node, an edge,
a checkpoint".

**You will learn**
- The node contract: *read state, return an update*, and why nodes mustn't mutate state
- What a **reducer** is and the data-loss bug you get without one
- How a **superstep** runs parallel branches and joins them
- What exactly a **checkpoint** stores, and how pause / resume / fork fall out of it

**Prerequisites:** `01_why_orchestration.ipynb` (or the idea that the agent loop
can't guarantee order, parallelism or durability).

## 1 · Why these particular pieces

Notebook 01 ended with four building blocks. Here's the job each one does and
the design question it forces:

| Block | Job | Design question |
|---|---|---|
| **State** | Holds all the work, explicitly | How are updates merged, especially from parallel branches? → **reducers** |
| **Node** | One named step | What may a node touch? → *read state, return an update* |
| **Edge** | What runs next | Fixed, chosen at runtime (**router**), or several at once (**fan-out**)? |
| **Checkpointer** | Makes progress durable | What must be saved so *any* process can continue? → state **and** position |

We'll build them in that order, then use the engine for seven demos.

In [ ]:
import json, sqlite3, time, threading, contextvars, pathlib
from concurrent.futures import ThreadPoolExecutor

## 2 · State, nodes and reducers

**The node contract.** A node is a function `node(state) -> update`. It gets a
*copy* of the state and returns a **partial update**, a dict of only the keys it
changes. It never modifies the state in place. Why so strict?

- **Parallel safety.** Two branches running at the same time must not write into
  one shared dict. Each returns its update, and the engine merges them.
- **Checkpoints.** The engine is the only thing that changes state, so it knows
  exactly when to save.
- **Testing.** A node is a pure-ish function: give it a state, check the update.

**Reducers.** When an update arrives, how is it merged into the state? By
default a new value **overwrites** the old one, which is right for things like
`status` or `draft`. For things like `notes`, where several branches each add an
item, you want to **append** instead. A *reducer* is the function that decides,
per key: `new_value = reducer(old_value, update_value)`.

In [ ]:
def append(old, new):
    '''Reducer: concatenate lists instead of overwriting.'''
    return (old or []) + list(new)


def merge(state, update, reducers):
    '''Apply one node's update to the state, key by key.'''
    state = dict(state)
    for key, value in update.items():
        if key in reducers:
            state[key] = reducers[key](state.get(key), value)
        else:
            state[key] = value                 # default reducer: overwrite
    return state


reducers = {'notes': append}
s = {'topic': 'retries', 'notes': ['a']}
s = merge(s, {'notes': ['b']}, reducers)         # appended
s = merge(s, {'topic': 'backoff'}, reducers)     # overwritten
print(s)

## 3 · The engine

Three pieces of vocabulary before the code:

- **Superstep.** The engine runs in rounds. Each round runs every node that is
  due *at the same time*, waits for all of them, merges their updates, then
  works out which nodes are due next. That's how fan-out/fan-in works: branches
  started in one round run in parallel, and a node they all point to runs once,
  in the next round.
- **Frontier.** The set of nodes due in the next superstep.
- **Checkpoint.** After every superstep the engine saves `(thread, step, state,
  frontier)`. That's the complete answer to "what must be saved to continue?":
  the data **and** the position.

```
                  ┌────────────── one superstep ──────────────┐
 frontier ──►  run every due node in parallel  ──► merge updates via reducers
    ▲                                                     │
    │                                                     ▼
    └── successors of the nodes that ran ◄── save checkpoint (state + frontier)
        (edges + routers; stop when empty)
```

First the **checkpointer**: a SQLite table with one row per `(thread, step)`.
A *thread* is one run of the workflow, identified by an id you choose, e.g. a
ticket number or a user session.

In [ ]:
class SqliteSaver:
    '''Durable checkpoints: one row per (thread, step).'''

    def __init__(self, path):
        pathlib.Path(path).parent.mkdir(exist_ok=True)
        self.db = sqlite3.connect(path, check_same_thread=False)
        self.db.execute('''CREATE TABLE IF NOT EXISTS checkpoints (
            thread TEXT, step INTEGER, state TEXT, frontier TEXT, paused TEXT,
            PRIMARY KEY (thread, step))''')

    def save(self, thread, step, state, frontier, paused=None):
        self.db.execute('INSERT OR REPLACE INTO checkpoints VALUES (?,?,?,?,?)',
                        (thread, step, json.dumps(state), json.dumps(frontier), json.dumps(paused)))
        self.db.commit()                        # ← on disk before the next step starts

    def _row(self, r):
        return {'step': r[0], 'state': json.loads(r[1]), 'frontier': json.loads(r[2]),
                'paused': json.loads(r[3])}

    def latest(self, thread):
        r = self.db.execute('SELECT step, state, frontier, paused FROM checkpoints '
                            'WHERE thread=? ORDER BY step DESC LIMIT 1', (thread,)).fetchone()
        return self._row(r) if r else None

    def history(self, thread):
        rows = self.db.execute('SELECT step, state, frontier, paused FROM checkpoints '
                               'WHERE thread=? ORDER BY step', (thread,)).fetchall()
        return [self._row(r) for r in rows]

Now **pausing**. A node that needs a human calls `interrupt(question)`. The
first time, that raises `Paused`: the engine stops and saves the question in the
checkpoint. When the human answers, the engine **re-runs that node from the
top**, and this time `interrupt()` *returns* the answer instead of raising.

That "re-runs from the top" detail matters, and LangGraph works the same way: any
code in the node *before* `interrupt()` runs twice. So put side effects *after*
the interrupt. (The `contextvars` lines are plumbing that hands the answer to
the right node when nodes run in threads; you can skim them.)

In [ ]:
START, END = 'START', 'END'
_NO_ANSWER = object()
_answer = contextvars.ContextVar('answer', default=_NO_ANSWER)


class Paused(Exception):
    '''Raised by interrupt() to stop the run and wait for a human.'''
    def __init__(self, question):
        self.question = question


def interrupt(question):
    '''Call inside a node. First run: pause. On resume: return the human's answer.'''
    answer = _answer.get()
    if answer is _NO_ANSWER:
        raise Paused(question)
    return answer

And the engine itself. Read the numbered comments in `invoke`: they are the
whole algorithm.

In [ ]:
class Graph:
    def __init__(self, reducers=None, saver=None, verbose=True):
        self.nodes, self.edges, self.routers = {}, {}, {}
        self.reducers = reducers or {}
        self.saver = saver
        self.verbose = verbose

    # ---- building the graph ------------------------------------------------
    def add_node(self, name, fn):
        self.nodes[name] = fn

    def add_edge(self, src, dst):
        '''Fixed edge. Several edges out of one node = fan-out (run in parallel).'''
        self.edges.setdefault(src, []).append(dst)

    def add_router(self, src, fn):
        '''Conditional edge: fn(state) returns the next node name (or a list of names).'''
        self.routers[src] = fn

    def _successors(self, node, state):
        nxt = list(self.edges.get(node, []))
        if node in self.routers:
            choice = self.routers[node](state)
            nxt += choice if isinstance(choice, list) else [choice]
        return nxt

    # ---- running it --------------------------------------------------------
    def _run_node(self, name, state, answer):
        ctx = contextvars.copy_context()
        if answer is not _NO_ANSWER:
            ctx.run(_answer.set, answer)                  # hand the human's answer to this node
        return ctx.run(self.nodes[name], dict(state))     # a COPY: nodes can't mutate state

    def invoke(self, inputs=None, thread='default', resume=_NO_ANSWER):
        '''inputs=dict → start a new run.  inputs=None → continue the thread's last checkpoint.'''
        # 1. Where are we? A fresh start, or the latest checkpoint of this thread.
        paused_at = None
        if inputs is not None:
            step, state = 0, merge({}, inputs, self.reducers)
            frontier = self._successors(START, state)
        else:
            ckpt = self.saver.latest(thread)
            step, state, frontier = ckpt['step'], ckpt['state'], ckpt['frontier']
            paused_at = (ckpt['paused'] or {}).get('node')

        while frontier and frontier != [END]:
            due = [n for n in frontier if n != END]
            if self.verbose:
                print(f'  step {step + 1}: run {due}')
            # 2. Run every due node in parallel. Only a paused node gets the resume answer.
            with ThreadPoolExecutor() as pool:
                futures = {n: pool.submit(self._run_node, n, state,
                                          resume if inputs is None and n == paused_at else _NO_ANSWER)
                           for n in due}
                try:
                    updates = {n: f.result() for n, f in futures.items()}
                except Paused as p:
                    # 3a. A node asked a human. Save WHERE we are and WHY, then stop.
                    #     The superstep is all-or-nothing: it will re-run on resume.
                    node = next(n for n, f in futures.items()
                                if isinstance(f.exception(), Paused))
                    if self.saver:
                        self.saver.save(thread, step, state, frontier, {'node': node, 'question': p.question})
                    if self.verbose:
                        print(f'  ⏸  paused in {node!r}: {p.question}')
                    return {**state, '__paused__': p.question}
            # 3. Merge every update into the state, key by key, via the reducers.
            for update in updates.values():
                state = merge(state, update or {}, self.reducers)
            # 4. The next frontier = the successors of every node that just ran (deduplicated).
            frontier = list(dict.fromkeys(s for n in due for s in self._successors(n, state)))
            step += 1
            # 5. Checkpoint: data AND position, on disk, before anything else happens.
            if self.saver:
                self.saver.save(thread, step, state, frontier)
            inputs, resume = None, _NO_ANSWER   # only the first superstep after a resume gets the answer
            paused_at = None
        return state

    # ---- time travel -------------------------------------------------------
    def fork(self, thread, step, new_thread, **changes):
        '''Copy checkpoint `step` of `thread` into `new_thread`, optionally editing the state.'''
        ckpt = next(c for c in self.saver.history(thread) if c['step'] == step)
        self.saver.save(new_thread, step, {**ckpt['state'], **changes}, ckpt['frontier'])

That's the engine. Now let's use it, one building block at a time.

## 4 · Demo: fixed edges and a router

A support-ticket workflow: classify the ticket, then **route** it. The router is
a plain function of the state. Here it's a keyword check, but it could just as
well call an LLM, which is how "let the model decide *here*" enters an
orchestrated system. The fixed edges guarantee that `reply` always comes last.

```
START → classify ──router──► billing ─┐
                        └──► tech   ──┴──► reply → END
```

In [ ]:
def classify(state):
    text = state['ticket'].lower()
    return {'category': 'billing' if any(w in text for w in ('refund', 'invoice', 'charge')) else 'tech'}

def billing(state):
    return {'answer': 'Refund issued to your card.'}

def tech(state):
    return {'answer': 'Please clear your cache and retry.'}

def reply(state):
    return {'sent': f"[{state['category']}] {state['answer']}"}

g = Graph()
for name, fn in [('classify', classify), ('billing', billing), ('tech', tech), ('reply', reply)]:
    g.add_node(name, fn)
g.add_edge(START, 'classify')
g.add_router('classify', lambda s: s['category'])   # ← conditional edge
g.add_edge('billing', 'reply')
g.add_edge('tech', 'reply')
g.add_edge('reply', END)

print(g.invoke({'ticket': 'I was charged twice, please refund'})['sent'])
print(g.invoke({'ticket': 'The app crashes on login'})['sent'])

## 5 · Demo: parallel branches, and why reducers exist

Three 1-second fetches fan out from `START` and fan back into `write`. They run
in **one** superstep, so the whole thing takes ~1s, not 3s. Each fetch returns
`{'notes': [...]}`, and the `append` reducer combines them.

In [ ]:
def fetcher(topic):
    def fetch(state):
        time.sleep(1.0)                       # a slow API
        return {'notes': [f'notes on {topic}']}
    return fetch

def write(state):
    return {'report': ' | '.join(sorted(state['notes']))}

def build_research(reducers, saver=None):
    g = Graph(reducers=reducers, saver=saver)
    for topic in ('checkpoints', 'retries', 'fan-out'):
        g.add_node(f'fetch_{topic}', fetcher(topic))
        g.add_edge(START, f'fetch_{topic}')          # fan-out: START → 3 nodes
        g.add_edge(f'fetch_{topic}', 'write')        # fan-in:  3 nodes → write
    g.add_node('write', write)
    g.add_edge('write', END)
    return g

t0 = time.perf_counter()
out = build_research({'notes': append}).invoke({'notes': []})
print(f'{time.perf_counter() - t0:.1f}s →', out['report'])

`write` ran **once**: all three fetches ran in step 1, and their shared successor
was deduplicated into step 2. That's fan-in.

Now remove the reducer and run the *same* graph. Each branch's update simply
**overwrites** `notes`:

In [ ]:
out = build_research(reducers={}).invoke({'notes': []})
print('report without a reducer →', out['report'])

Two of the three notes silently vanished. The last branch to be merged won, and
no error was raised. This is the most common bug in parallel workflows, and it's
why every framework makes you declare how shared keys combine (LangGraph:
`Annotated[list, operator.add]`). **Rule: any key written by more than one
parallel branch needs a reducer.**

## 6 · Demo: a cycle with a guard

Loops are just edges pointing backwards: write a draft, review it, and if it's
not good enough go back and rewrite. Every cycle needs a **guard**, a hard
limit on iterations, or a model that is never satisfied will loop forever. The
guard is ordinary state (`revisions`) checked by the router.

In [ ]:
def draft(state):
    n = state.get('revisions', 0) + 1
    return {'revisions': n, 'draft': f'draft v{n}'}

def review(state):
    return {'approved': state['revisions'] >= 3}    # this reviewer is hard to please

MAX_REVISIONS = 5

g = Graph()
g.add_node('draft', draft)
g.add_node('review', review)
g.add_edge(START, 'draft')
g.add_edge('draft', 'review')
g.add_router('review', lambda s: END if s['approved'] or s['revisions'] >= MAX_REVISIONS else 'draft')

print(g.invoke({})['draft'])

## 7 · Demo: checkpoints, a crash, and resuming

Now the durability story from notebook 01, on the real engine. We give the
research graph a `SqliteSaver`, and make `write` crash the first time. Because a
checkpoint was saved after step 1, resuming skips the fetches entirely.

`ProcessCrash` subclasses `BaseException` to model the process dying: nothing
inside the engine catches it.

In [ ]:
class ProcessCrash(BaseException):
    pass

DB = 'checkpoints/02_engine.db'
pathlib.Path(DB).unlink(missing_ok=True)       # start fresh each time this cell runs

fetch_count = {'n': 0}
crash_once = {'armed': True}

def counting_fetcher(topic):
    def fetch(state):
        fetch_count['n'] += 1
        time.sleep(0.3)
        return {'notes': [f'notes on {topic}']}
    return fetch

def crashing_write(state):
    if crash_once['armed']:
        crash_once['armed'] = False
        raise ProcessCrash('killed while writing')
    return {'report': ' | '.join(sorted(state['notes']))}

def build_durable(saver):
    g = Graph(reducers={'notes': append}, saver=saver)
    for topic in ('checkpoints', 'retries', 'fan-out'):
        g.add_node(f'fetch_{topic}', counting_fetcher(topic))
        g.add_edge(START, f'fetch_{topic}')
        g.add_edge(f'fetch_{topic}', 'write')
    g.add_node('write', crashing_write)
    g.add_edge('write', END)
    return g

print('process 1:')
try:
    build_durable(SqliteSaver(DB)).invoke({'notes': []}, thread='report-42')
except ProcessCrash as err:
    print('  💥', err)

print('process 2 (new engine, new saver, same database file):')
out = build_durable(SqliteSaver(DB)).invoke(None, thread='report-42')
print('\nreport:', out['report'])
print('fetches executed in total:', fetch_count['n'], '(3, not 6)')

Look at what "process 2" had: no variables from process 1, just the thread id
and the database file. That was enough, because the checkpoint held the state
**and** the frontier (`['write']`). This is also why state must be
JSON-serialisable: `json.dumps` in `save()` would fail on a client object or an
open file.

**Threads keep runs apart.** A thread id is simply a key. Two tickets, two
threads, zero interference:

In [ ]:
router_graph = Graph(saver=SqliteSaver(DB), verbose=False)
for name, fn in [('classify', classify), ('billing', billing), ('tech', tech), ('reply', reply)]:
    router_graph.add_node(name, fn)
router_graph.add_edge(START, 'classify')
router_graph.add_router('classify', lambda s: s['category'])
router_graph.add_edge('billing', 'reply')
router_graph.add_edge('tech', 'reply')
router_graph.add_edge('reply', END)

router_graph.invoke({'ticket': 'refund my invoice'}, thread='ticket-1')
router_graph.invoke({'ticket': 'login is broken'}, thread='ticket-2')
for t in ('ticket-1', 'ticket-2'):
    print(t, '→', router_graph.saver.latest(t)['state']['sent'])

## 8 · Demo: pause for a human, resume in another process

A publishing workflow: draft, then **ask a human**, then publish or discard.
The `approve` node calls `interrupt(...)`. The engine saves a checkpoint that
records *who* paused and *what they asked*, and `invoke` simply returns. No
thread is blocked and no process has to stay alive.

Hours later, possibly in a new process, we call `invoke(None, thread, resume=answer)`.
Watch the `print` inside `approve`: it runs **twice**, once before the pause and
once on resume. That's the "node re-runs from the top" rule.

In [ ]:
def write_post(state):
    return {'draft': f"Post about {state['topic']}"}

def approve(state):
    print(f"    (approve node running, draft = {state['draft']!r})")
    decision = interrupt({'question': 'Publish this?', 'draft': state['draft']})
    return {'decision': decision}              # side effects belong AFTER interrupt()

def publish(state):
    return {'status': f"published: {state['draft']}"}

def discard(state):
    return {'status': 'discarded'}

def build_publisher(saver):
    g = Graph(saver=saver)
    for name, fn in [('write_post', write_post), ('approve', approve),
                     ('publish', publish), ('discard', discard)]:
        g.add_node(name, fn)
    g.add_edge(START, 'write_post')
    g.add_edge('write_post', 'approve')
    g.add_router('approve', lambda s: 'publish' if s['decision'] == 'yes' else 'discard')
    g.add_edge('publish', END)
    g.add_edge('discard', END)
    return g

print('Monday, process 1:')
out = build_publisher(SqliteSaver(DB)).invoke({'topic': 'reducers'}, thread='post-7')
print('  returned:', out['__paused__'])

print('\nTuesday, process 2, the reviewer says yes:')
out = build_publisher(SqliteSaver(DB)).invoke(None, thread='post-7', resume='yes')
print('  status:', out['status'])

## 9 · Demo: time travel, inspect and fork

Every superstep left a checkpoint, so the full history of a thread is just a
query. You can look at the state *at any step*, which answers "what did the
workflow know when it made that decision?" You can also **fork**: copy an old
checkpoint into a new thread, change something, and run forward from there
without touching the original run.

In [ ]:
publisher = build_publisher(SqliteSaver(DB))
for c in publisher.saver.history('post-7'):
    paused = f"  ⏸ {c['paused']['question']['question']}" if c['paused'] else ''
    print(f"step {c['step']}: next={c['frontier']}  state={c['state']}{paused}")

In [ ]:
# Fork from step 1 (draft written, approval pending) into a new thread,
# with a different draft, and let a different reviewer decide.
publisher.fork('post-7', step=1, new_thread='post-7-alt', draft='A shorter post about reducers')

out = publisher.invoke(None, thread='post-7-alt')          # runs approve → pauses again
out = publisher.invoke(None, thread='post-7-alt', resume='no')
print('fork  :', out['status'])
print('orig  :', publisher.saver.latest('post-7')['state']['status'], '(untouched)')

## 10 · What you built, in every framework's vocabulary

| Our engine | LangGraph | Elsewhere in this phase |
|---|---|---|
| `Graph`, `add_node`, `add_edge` | `StateGraph`, `add_node`, `add_edge` | MS Agent Framework: `WorkflowBuilder`, executors, edges |
| `add_router` | `add_conditional_edges` | CrewAI Flows: `@router`; ADK: agent transfer |
| edges out of `START` to several nodes | same (fan-out), or `Send` for a dynamic number | ADK: `ParallelAgent` |
| `reducers={'notes': append}` | `Annotated[list, operator.add]` in the state schema | CrewAI Flows: you update the state yourself |
| `SqliteSaver`, `thread` | checkpointer (`SqliteSaver`, …), `thread_id` | Temporal: event history; ADK / OpenAI SDK: sessions |
| `interrupt()`, `resume=` | `interrupt()`, `Command(resume=...)` | Temporal: signals |
| `history`, `fork` | `get_state_history`, `update_state` + invoke from a checkpoint | |
| superstep loop | the Pregel runtime | |

## 11 · Trade-offs and limits of this design

- **Supersteps are simple but lock-step.** A fast branch waits for the slowest
  branch in the same round. Most frameworks accept this for predictability.
- **Fan-in by deduplication only works for branches of equal length.** If one
  branch has two nodes and the other one, the join node would run twice.
  Frameworks add explicit "wait for all of these" edges (in LangGraph,
  `add_edge(['a', 'b'], 'join')`).
- **A pause discards the rest of its superstep**, which then re-runs on resume.
  Production engines save the finished branches' results ("pending writes") so
  they don't repeat.
- **Checkpointing every step costs I/O.** It's a write per superstep: cheap
  compared to an LLM call, but not free for huge states. Keep state lean and
  store big blobs (files, documents) by reference.
- **Durable ≠ exactly-once.** If the process dies *during* a node, that node
  runs again on resume. Nodes with side effects (payments, emails) must be
  **idempotent**, e.g. check "already sent?" before sending.

## 12 · Common mistakes

- **Mutating `state` inside a node** instead of returning an update. It breaks
  parallel branches and makes checkpoints lie.
- **No reducer on a key written by parallel branches**: silent data loss (§5).
- **A cycle without a guard**: an infinite loop that also burns tokens.
- **Side effects before `interrupt()`**: they happen twice (§8).
- **Unserialisable values in state**: the first checkpoint fails.
- **Reusing a thread id for an unrelated run**: you resume someone else's state.

## 13 · Check yourself

<details><summary><b>1.</b> Why does a node return an update instead of modifying the state it's given?</summary>

So the engine is the only writer. Parallel branches can't clobber each other, the engine knows exactly when to checkpoint, and each node can be tested as "state in, update out". (Our engine even passes a copy, so mutation would simply be lost.)
</details>

<details><summary><b>2.</b> Three parallel branches each return <code>{'notes': [x]}</code> and there's no reducer. What ends up in <code>notes</code>, and why is this bug dangerous?</summary>

Only one branch's list: whichever update is merged last overwrites the others. It's dangerous because nothing fails. The run "succeeds" with missing data. Fix: declare an append reducer for `notes`.
</details>

<details><summary><b>3.</b> List exactly what a checkpoint must contain for a brand-new process to continue a run.</summary>

The thread id and step (which run, how far), the full **state**, and the **frontier** (which nodes run next). For a paused run it must also record *which node* paused and *what it asked*.
</details>

<details><summary><b>4.</b> In §8 the <code>print</code> in <code>approve</code> appeared twice. Why, and what rule follows?</summary>

On resume the engine re-runs the paused node from the top. This time `interrupt()` returns the answer instead of raising. So anything before `interrupt()` executes twice. Rule: put side effects after the interrupt (or make them idempotent).
</details>

<details><summary><b>5.</b> A process crashes <em>halfway through</em> a node that charges a credit card. After resume, what happens, and how do you make it safe?</summary>

The last checkpoint is from before that node, so the node runs again and the card could be charged twice. Make the node **idempotent**: use an idempotency key (for example, the thread id plus the step), or check "already charged?" before charging. Durability gives at-least-once execution, not exactly-once.
</details>

<details><summary><b>6.</b> When would you write a router as an LLM call instead of an <code>if</code>?</summary>

When the decision needs judgement that code can't express cheaply, such as understanding free text, intent, or quality. If the decision is a simple rule on structured state (a flag, a count, a category), use code: it's free, instant and deterministic.
</details>

## Takeaway

An orchestration engine is a loop over **supersteps**: run the due nodes (in
parallel), merge their updates through **reducers**, follow **edges and
routers** to the next frontier, and **checkpoint** state + frontier. Pause,
resume, crash recovery, separate threads and time travel all come from that one
checkpoint. Every framework in Phase 5 is this machine with a nicer API.

Next → `03_workflows_vs_agents.ipynb`: *how much* control to give the model:
the spectrum from fixed workflows to autonomous agents.